# Program Parser 完整指南

Program Parser 用于解析代码、提取函数和类、检查语法错误。

## 目录
1. [CodeParser 类](#1-CodeParser-类)
2. [解析结果类](#2-解析结果类)
3. [使用示例](#3-使用示例)
4. [提取代码技巧](#4-提取代码技巧)

## 1. CodeParser 类

In [ ]:
from algodisco.toolkit.program_parser import CodeParser, has_syntax_error

# 创建解析器 - 目前支持 python
parser = CodeParser("python")

print(f"语言: {parser.language_name}")

## 2. 解析结果类

In [ ]:
# 解析结果包含三个类:
# - CodeBlock: 代码片段（非函数/类）
# - Function: 函数定义
# - Class: 类定义
# - Program: 整个程序

from algodisco.toolkit.program_parser import CodeBlock, Function, Class, Program

print("CodeBlock: code, indent")
print("Function: name, pre_name, post_name, body, footer, docstring, is_async")
print("Class: name, pre_name, post_name, footer, body, docstring")
print("Program: scripts, functions, classes, elements")

## 3. 使用示例

In [ ]:
# 解析代码

code = """
import os

def add(a, b):
    '''Add two numbers'''
    return a + b

class Calculator:
    def __init__(self):
        self.result = 0
    
    def add(self, x):
        self.result += x
        return self.result
"""

parser = CodeParser("python")
program = parser.parse(code)

print(f"解析结果:")
print(f"  函数数量: {len(program.functions)}")
print(f"  类数量: {len(program.classes)}")
print(f"  代码片段数量: {len(program.scripts)}")

In [ ]:
# 访问函数

for func in program.functions:
    print(f"\n函数: {func.name}")
    print(f"  is_async: {func.is_async}")
    print(f"  docstring: {func.docstring}")
    print(f"  body: {func.body[:50]}...")

In [ ]:
# 访问类

for cls in program.classes:
    print(f"\n类: {cls.name}")
    print(f"  方法数量: {len(cls.body)}")
    for item in cls.body:
        if isinstance(item, Function):
            print(f"    - 方法: {item.name}")

In [ ]:
# 重建代码

reconstructed = str(program)
print("重建的代码:")
print(reconstructed)

## 4. 语法检查

In [ ]:
# 检查语法错误

# 正确的代码
valid_code = "def test():\n    pass"
print(f"有效代码: {has_syntax_error(valid_code, 'python')}")

# 有错误的代码
invalid_code = "def test(:\n    pass"
print(f"无效代码: {has_syntax_error(invalid_code, 'python')}")

In [ ]:
# 在解析时自动检查语法

try:
    program = parser.parse("def broken(: pass")
except RuntimeError as e:
    print(f"捕获到语法错误: {e}")

## 5. 提取代码技巧

In [ ]:
# 技巧1: 从LLM响应中提取代码

import re

def extract_code_from_response(response: str, language: str = "python") -> str:
    """从LLM响应中提取代码块"""
    # 匹配 ```python ... ``` 或 ``` ... ```
    pattern = rf"```(?:{language})?\s*\n(.*?)\n```"
    match = re.search(pattern, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()

# 测试
llm_response = """
Here's the code:
```python
def sort_list(lst):
    return sorted(lst)
```
"""

code = extract_code_from_response(llm_response)
print(f"提取的代码:\n{code}")

In [ ]:
# 技巧2: 检查并清理代码

def validate_and_clean_code(code: str, language: str = "python") -> tuple[bool, str]:
    """验证并清理代码"""
    # 提取代码
    code = extract_code_from_response(code, language)
    
    # 检查语法
    if has_syntax_error(code, language):
        return False, ""
    
    return True, code

# 测试
valid_code = """
```python
def test(): pass
```
"""
ok, cleaned = validate_and_clean_code(valid_code)
print(f"有效: {ok}, 代码: {cleaned}")

invalid_code = """
```python
def test(: pass
```
"""
ok, cleaned = validate_and_clean_code(invalid_code)
print(f"有效: {ok}, 代码: {cleaned}")

## 总结

| 类/函数 | 说明 |
|---------|------|
| `CodeParser` | 解析器类 |
| `CodeParser.parse()` | 解析代码 |
| `has_syntax_error()` | 检查语法错误 |
| `CodeBlock` | 代码片段 |
| `Function` | 函数定义 |
| `Class` | 类定义 |
| `Program` | 整个程序 |
| `str(program)` | 重建代码 |